In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# ★ 보고 기간 설정
start_date = '2026-06-01'
end_date = '2026-06-30'

In [ ]:
# 판매이력 불러오기
sales_file = "6.2.1 Delivery Status(List Only).xlsx"

# 피벗/요약 시트(예: Sheet1)가 앞에 있어도 실제 데이터 시트를 골라 읽기
sheet_names = pd.ExcelFile(sales_file).sheet_names
data_sheet = next((s for s in sheet_names if s.startswith("6.2.1")), sheet_names[0])

df_s = pd.read_excel(sales_file, sheet_name=data_sheet, skiprows=6)
df_s = df_s.drop("Unnamed: 0", axis=1, errors="ignore")

mask = ((df_s['Delivery Date.1'] >= start_date ) & (df_s['Delivery Date.1'] <= end_date))
df_s = df_s.loc[mask]

In [ ]:
# 구매이력 불러오기 (9.1 AP Invoice Status)
# AP Invoice = 매입세금계산서 → 공급업체에 지불한 구매단가(Unit Price)를 구매원가로 사용
purchase_file = "9.1 AP Invoice Status.xlsx"

# 데이터 시트 자동 선택 (헤더는 8번째 행 → skiprows=7)
p_sheets = pd.ExcelFile(purchase_file).sheet_names
p_sheet = next((s for s in p_sheets if s.startswith("9.1")), p_sheets[0])

df_p = pd.read_excel(purchase_file, sheet_name=p_sheet, skiprows=7)
df_p = df_p.rename(columns={"Unit Price (IDR)": "P_Price(IDR)"})  # 열이름 변경

# Item Code별로 여러 매입건이 있으므로, A/P Inv Date 기준 '최신 구매단가'만 남김
df_p = df_p[["Item Code", "A/P Inv Date", "P_Price(IDR)"]].dropna(subset=["Item Code"])
df_p["A/P Inv Date"] = pd.to_datetime(df_p["A/P Inv Date"])
df_p = (df_p.sort_values("A/P Inv Date")
             .drop_duplicates("Item Code", keep="last"))
df_p = df_p[["Item Code", "P_Price(IDR)"]]

In [ ]:
# 1. Row_data : 구매단가 합치기 vlookup... 최종가격(Item Code별 최신단가)
df = df_s.merge(df_p, how='left', on="Item Code").dropna(subset=["P_Price(IDR)"])
df["P_Amount(IDR)"] = df["P_Price(IDR)"] * df["Q'ty"]
df["Margin(IDR)"] = df["Discounted Amount (IDR)"] - df["P_Amount(IDR)"]
df["Margin(%)"] = df["Margin(IDR)"] / df["Discounted Amount (IDR)"]

df['Brand'] = df['Brand'].fillna("ETC") # 없는 브랜드 채우기
# df['Type 2'] = df['Type 2'].replace('Others','Others 1') # 동일명칭 변경
df['Type 3'] = df['Type 3'].replace('Others','ETC')
# df.loc[df['Type 2'] == 'Others 1']
# df.to_clipboard()

In [ ]:
# 2 업체별 마진분석(번호	고객명	판매금액(IDR)	마진금액(IDR)	마진(%)) 매출점유율, 마진점유율
df_cust = df.groupby("Buyer")[["Discounted Amount (IDR)","Margin(IDR)"]].agg(sum)

df_cust["Margin(%)"] = df_cust["Margin(IDR)"] / df_cust["Discounted Amount (IDR)"]

df_cust.loc["TOTAL", :] = df_cust.sum() # 합계넣기
df_cust.loc["TOTAL", "Margin(%)"] = df_cust.loc["TOTAL","Margin(IDR)"] / df_cust.loc["TOTAL","Discounted Amount (IDR)"] # 총마진 평균으로 수정

df_cust["Sales Ratio(%)"] = df_cust["Discounted Amount (IDR)"] / df_cust.loc["TOTAL","Discounted Amount (IDR)"]
df_cust["Margin Ratio(%)"] = df_cust["Margin(IDR)"] / df_cust.loc["TOTAL","Margin(IDR)"]

df_cust = df_cust.sort_values(["Margin(IDR)"], ascending=False).reset_index()
# df_cust.info()

In [ ]:
# 3. 상품별 마진분석(번호	DESC	판매수량	판매금액(IDR)	마진금액(IDR)	마진(%))
df_prod = df.groupby(["Type 3", "Type 2"])[["Q'ty","Discounted Amount (IDR)","Margin(IDR)"]].agg(sum)
df_prod["Margin(%)"] = df_prod["Margin(IDR)"] / df_prod["Discounted Amount (IDR)"]

df_prod.loc[("TOTAL", "ALL"), :] = df_prod.sum() # 튜플형태로 합계넣기
df_prod.loc[("TOTAL", "ALL"), "Margin(%)"] = df_prod.loc[("TOTAL", "ALL"),"Margin(IDR)"] / df_prod.loc[("TOTAL", "ALL"),"Discounted Amount (IDR)"] # 총마진 평균으로 수정

df_prod["Sales Ratio(%)"] = df_prod["Discounted Amount (IDR)"] / df_prod.loc[("TOTAL", "ALL"),"Discounted Amount (IDR)"]
df_prod["Margin Ratio(%)"] = df_prod["Margin(IDR)"] / df_prod.loc[("TOTAL", "ALL"),"Margin(IDR)"]

df_prod = df_prod.sort_values(["Margin(IDR)"], ascending=False).reset_index()
# df_prod

In [ ]:
# 4. 아이템별 마진분석(번호	상품	판매수량	판매금액(IDR)	마진금액(IDR)	마진(%))
df_item = df.groupby(["Description"])[["Q'ty","Discounted Amount (IDR)","Margin(IDR)"]].agg(sum)
df_item["Margin(%)"] = df_item["Margin(IDR)"] / df_item["Discounted Amount (IDR)"]

df_item.loc["TOTAL", :] = df_item.sum() # 합계넣기
df_item.loc["TOTAL", "Margin(%)"] = df_item.loc["TOTAL","Margin(IDR)"] / df_item.loc["TOTAL","Discounted Amount (IDR)"] # 총마진 평균으로 수정

df_item["Sales Ratio(%)"] = df_item["Discounted Amount (IDR)"] / df_item.loc["TOTAL","Discounted Amount (IDR)"]
df_item["Margin Ratio(%)"] = df_item["Margin(IDR)"] / df_item.loc["TOTAL","Margin(IDR)"]


df_item = df_item.sort_values(["Margin(IDR)"], ascending=False).reset_index()
# df_item.to_clipboard()
# df_item

In [ ]:
# 5. 브랜드별 마진분석(번호	브랜드	판매금액(IDR)	마진금액(IDR)	마진(%))
df_bran = df.groupby(["Brand"])[["Discounted Amount (IDR)","Margin(IDR)"]].agg(sum)
df_bran["Margin(%)"] = df_bran["Margin(IDR)"] / df_bran["Discounted Amount (IDR)"]

df_bran.loc["TOTAL", :] = df_bran.sum() # 합계넣기
df_bran.loc["TOTAL", "Margin(%)"] = df_bran.loc["TOTAL","Margin(IDR)"] / df_bran.loc["TOTAL","Discounted Amount (IDR)"] # 총마진 평균으로 수정

df_bran["Sales Ratio(%)"] = df_bran["Discounted Amount (IDR)"] / df_bran.loc["TOTAL","Discounted Amount (IDR)"]
df_bran["Margin Ratio(%)"] = df_bran["Margin(IDR)"] / df_bran.loc["TOTAL","Margin(IDR)"]

df_bran = df_bran.sort_values(["Margin(IDR)"], ascending=False).reset_index()
# df_bran

In [ ]:
# 열이름 변경
list_df = [df_cust, df_prod, df_item, df_bran]

for lst in list_df:
    lst.rename(columns={"Discounted Amount (IDR)":"Sales(IDR)"}, inplace=True) # 열이름 변경

df_prod = df_prod.rename(columns={"Type 3":"Product", "Type 2":"Type"}) # 열이름 변경

In [ ]:
# 콤마 넣기 함수
def int_num(x):
    return '{:,}'.format(int(x))

# 퍼센트 넣기
def pct(x):
    return '{:.1f}'.format(x*100)+"%"

df_prod["Q'ty"] = df_prod["Q'ty"].astype("int").apply(lambda x : '{:,}'.format(x))
df_item["Q'ty"] = df_item["Q'ty"].astype("int").apply(lambda x : '{:,}'.format(x))

list_df = [df_cust, df_prod, df_item, df_bran]

for lst in list_df:
    # 콤마 넣기
    lst["Sales(IDR)"] = lst["Sales(IDR)"].apply(int_num)
    lst["Margin(IDR)"] = lst["Margin(IDR)"].apply(int_num)

    # 퍼센트 넣기
    lst["Margin(%)"] = lst["Margin(%)"].apply(pct)
    lst["Sales Ratio(%)"] = lst["Sales Ratio(%)"].apply(pct)
    lst["Margin Ratio(%)"] = lst["Margin Ratio(%)"].apply(pct)

In [ ]:
# 연월 설정
import datetime

dates = df["Delivery Date.1"].iloc[0].strftime("%Y-%b-%d")
year = dates[0:4]
month = dates[5:8]                                       # 표시용 월 이름 (예: Jun)
month_num = df["Delivery Date.1"].iloc[0].strftime("%m") # 파일명용 월 숫자 (예: 06)

current_date = datetime.datetime.now().strftime("%b-%d, %Y")

In [ ]:
# 엑셀 출력
import openpyxl

#1. 파일 생성 (PDF와 동일한 파일명 형태, 월은 숫자)
writer=pd.ExcelWriter(f'sales analysis report {year}-{month_num}.xlsx', engine='openpyxl')

#2. 생성 파일에 시트명 지정 후 dataframe에 저장한 결과값 넣기
df.to_excel(writer, sheet_name='1.Row_Data') # 1
df_cust.to_excel(writer, sheet_name='2.Margin by Customer') # 2.
df_prod.to_excel(writer, sheet_name='3.Margin by Product') # 3.
df_item.to_excel(writer, sheet_name='4.Margin by Item') # 4.
df_bran.to_excel(writer, sheet_name='5.Margin by Brand') # 5.


#3. 작성 완료 후 파일 저장
writer._save() # save() 안되면 _save()로 저장

In [ ]:
# pdf 출력
# pip install fpdf2를 설치해야 에러가 없음
from fpdf import FPDF

# ★ Unicode 폰트 경로 (Windows 기본 - 맑은 고딕)
FONT_REG  = r"C:\Windows\Fonts\malgun.ttf"     # Regular
FONT_BOLD = r"C:\Windows\Fonts\malgunbd.ttf"   # Bold


class PDF(FPDF):
    def header(self):
        self.set_font('Malgun', 'B', 20)
        self.set_text_color(32,32,32)
        self.set_fill_color(240,255,255) # background = azure
        self.set_line_width(0.4) # thickness of border

        self.cell(0, 20, border=1, align='C', fill=1) # border
        self.image('ASCENDO_Blue.png', 15, 14, 50) # image
        self.cell(-150, 20, f'SALES ANALYSIS for {month} {year}',
                  new_x="LMARGIN", new_y="NEXT", align='C')
        self.ln(10)

    def footer(self):
        self.set_y(-15)
        self.set_font('Malgun', '', 10)  # 맑은고딕에 Italic 없음 → Regular
        self.set_text_color(169,169,169) # font color dark grey
        self.cell(0, 10, f'page {self.page_no()} / {{nb}}', align='C')


pdf = PDF('P', 'mm', 'A4')

# ★ Unicode 폰트 등록 (add_page 전에 수행)
pdf.add_font('Malgun', '',  FONT_REG)
pdf.add_font('Malgun', 'B', FONT_BOLD)

pdf.alias_nb_pages()
pdf.set_auto_page_break(auto=True, margin=15)

pdf.set_font("Malgun", size=7.5)
pdf.add_page()
line_height = pdf.font_size * 2


# 보고서 작성일 추가
pdf.cell(0, line_height, f'Date : {current_date}',
         new_x="LMARGIN", new_y="NEXT", align='R')
pdf.cell(0, line_height, 'Made : Jonghwan SEO',
         new_x="LMARGIN", new_y="NEXT", align='R')


# 데이터프레임으로 PDF작성
list_df = [df_bran, df_prod, df_cust, df_item]
list_df_name = ['1. MARGIN by BRANDs','2. MARGIN by PRODUCTs','3. MARGIN by CUSTOMERs','4. MARGIN by ITEMs']


for lst, lst_name in zip(list_df, list_df_name):
    # 1)제목 출력
    def header_1():
        pdf.set_x(8)
        pdf.set_font("Malgun", style="B")
        pdf.cell(150, line_height*2, lst_name)  # 기본칸 * 3
        pdf.ln(line_height)
    header_1()

    # fpdf에 맞게 튜플형으로 변경: str, tuple
    lst = lst.astype('str')
    TABLE_COL_NAMES = tuple(lst.columns)

    t1 = lst.values.tolist()
    table = []
    for i in range(len(t1)):
        x = tuple(t1[i])
        table.append(x)

    TABLE_DATA = tuple(table)

    # 표 테이블 렌더링
    pdf.ln(line_height)
    col_width = pdf.epw / (len(TABLE_COL_NAMES)+1)  # distribute content evenly # 기본칸 + 2

    def render_table_header():
        pdf.set_font("Malgun", style="B")  # enabling bold text
        pdf.set_fill_color(220,220,220) # font color dark grey

        for col_name in TABLE_COL_NAMES:
            if col_name == TABLE_COL_NAMES[0]:
                pdf.cell(col_width * 2, line_height, col_name, border=1, align='C', fill=1) # 기본칸 * 3
            else:
                pdf.cell(col_width, line_height, col_name, border=1, align='C', fill=1)
        pdf.ln(line_height)
        pdf.set_font("Malgun", style="")  # disabling bold text

    render_table_header()

    for row in TABLE_DATA:
        if pdf.will_page_break(line_height):
            render_table_header()
        for datum in row:
            if datum == row[0]:
                pdf.cell(col_width * 2, line_height, datum, border=1)  # 기본칸 * 3
            else:
                pdf.cell(col_width, line_height, datum, border=1, align='C')
        pdf.ln(line_height)
# 파일 저장, 파일명-연도-월(숫자)
pdf.output(f"sales analysis report {year}-{month_num}.pdf")

# 파일 저장 정상실행 알림 팝업창 실행
print(f"PDF file 'sales analysis report {year}-{month_num}.pdf' has been created")